In [ ]:
import numpy as np
import pandas as pd

In [ ]:
cust = pd.read_csv(r"data/Customer.csv")
cust.head()

In [ ]:
prod = pd.read_csv(r"data/prod_cat_info.csv")
prod.head()

In [ ]:
trans = pd.read_csv(r"data/Transactions.csv")
trans.head()

### <center> Data Familiarization </center>

##### Dimensions of the dataset

In [ ]:
print("Customer:",cust.shape)
print("\nProduct:",prod.shape)
print("\nTransactions:",trans.shape)

##### Getting concised summary of dataset

In [ ]:
cust.info()

In [ ]:
prod.info()

In [ ]:
trans.info()

##### Checking missing values

In [ ]:
cust.isnull().sum()

In [ ]:
prod.isnull().sum()

In [ ]:
trans.isnull().sum()

##### Checking duplicate values

In [ ]:
cust.duplicated().sum()

In [ ]:
prod.duplicated().sum()

In [ ]:
trans.duplicated().sum()

In [ ]:
trans[trans.duplicated()]

##### Changing datatype of date columns from string to datetime

In [ ]:
cust["DOB"] = pd.to_datetime(cust["DOB"],format="%d-%m-%Y")

In [ ]:
# To check whether it changed or not

cust["DOB"].dtype

In [ ]:
trans["tran_date"] = pd.to_datetime(trans["tran_date"],format="%d-%m-%Y")

In [ ]:
# To check whether it changed or not

trans["tran_date"].dtype

##### Getting summary of numerical columns

In [ ]:
trans.describe().T

##### Only checked for transaction dataset because it does not make sense for other two dataset.

#### Conclusion

We explored the three datasets: Customer, Transactions, and Product Hierarchy. These datasets contain information about customers, their purchases, and product categories. We checked the size of each dataset, its columns, data types, missing values, and basic statistics to understand the data better. These checks will help us prepare clean and reliable data for the next stages of the retail analytics project. 

### <center> Transaction Health & Data Integrity </center>

#### 1. The Phantom Transactions

In [ ]:
print("No. of transactions having negative monetary values:",trans[trans.total_amt<0].shape[0]) 

In [ ]:
# Calculating percentage

neg_count = (trans.total_amt<0).sum()
neg_percent = (neg_count/trans.shape[0]) * 100

print("Negative transactions:", neg_count)
print("\nTotal transactions:", trans.shape[0])
print("\nPercentage of negative transactions:", round(neg_percent, 2), "%")

In [ ]:
# Checking quantity distribution for negative-amount

trans[trans["total_amt"] < 0]["Qty"].value_counts()

In [ ]:
# Checking negative amounts with non-negative quanitites

trans[(trans["total_amt"] < 0) & (trans["Qty"] >= 0)].shape[0]

In [ ]:
# Checking negative transactions by store-type

trans[trans["total_amt"] < 0]["Store_type"].value_counts()

#### Conclusion

Out of 23,053 transactions, **2,177 transactions (9.44%) have negative amounts**. The negative transactions appear to be related to product returns, as their quantities are also negative rather than simple data-entry errors. Among the store types, e-Shop has the highest number of negative transactions (882), followed by MBR (451), Flagship store (432), and TeleShop (412). Based on the available evidence, there is no clear indication that these transactions are data-entry errors; they appear consistent with returns. 

#### 2. How Far Back Does Our Memory Go?

In [ ]:
# Findind date range

print("Earliest Transaction Date:", trans["tran_date"].dt.date.min())
print("Latest Transaction Date:", trans["tran_date"].dt.date.max())

In [ ]:
# To get no of transactions occured in each year.

trans["tran_date"].dt.year.value_counts().sort_index().reset_index()

In [ ]:
# To get no of trasactions month-wise

monthly_trans = trans["tran_date"].dt.month.value_counts().sort_index().reset_index()

monthly_trans.columns = ["Month", "Transaction_Count"]
monthly_trans.index = monthly_trans.index +1

monthly_trans

In [ ]:
# Checking no of transactions in each year and month

trans.groupby(trans["tran_date"].dt.to_period("M")).size()   # "M" for monthly frequency

In [ ]:
# No of years (year-span)

print("No. of years:",trans["tran_date"].dt.year.max() - trans["tran_date"].dt.year.min() + 1)

#### Conclusion

The transaction dataset contains 23,053 transaction records and spans a difference of 4 years between the earliest and latest transaction years. This provides a historical basis for analyzing customer purchasing behavior and identifying trends over time. However, the date range alone does not confirm that the data is evenly distributed across all months and years. Monthly transaction counts should also be considered to identify potential gaps or periods with limited data. If the data coverage is sufficiently consistent, the company can use it to compare purchasing patterns across years, including same-month comparisons, to support marketing and sales decisions.

### <center> Customer Geography & Demographics </center>

#### 3. Where Should We Open the Next Store?

In [ ]:
print("No. of customers:",cust.shape[0])

In [ ]:
# Checking the columns in the dataset to perform merge operation.

print(trans.columns)
print()
print(cust.columns)
print()
print(prod.columns)

In [ ]:
# Merging customer and transaction dataset

retail = pd.merge(trans,cust,left_on="cust_id",right_on="customer_Id",how="left")

retail.head()

In [ ]:
# 1st way

cust.groupby("city_code").size().sort_values(ascending=False)

In [ ]:
# 2nd way

cust["city_code"].value_counts()

In [ ]:
# To check City-wise transaction volume

retail.groupby("city_code")["transaction_id"].nunique().sort_values(ascending=False)

In [ ]:
# To calculate revenue generated by each-city

retail.groupby("city_code")["total_amt"].sum().sort_values(ascending= False)

In [ ]:
# To calculate average transaction value in each city

retail.groupby("city_code")["total_amt"].mean().sort_values(ascending=False)

In [ ]:
# To calculate city-wise quantity purchased

retail.groupby("city_code")["Qty"].sum().sort_values(ascending=False)

#### Conclusion

The analysis of 5,647 customers across 10 cities shows that City 3 has the highest customer base (595), transaction volume (2,196) and revenue (5,165,199.85). City 4 records the highest purchase quantity (5,954), while City 9 has the highest average transaction value (2,157.55). City 6 consistently records the lowest values across the analysed metrics. Based on these findings, City 3 appears to be a promising market for further expansion assessment. 

#### 4: Who Buys What? Gender and Category Preferences

In [ ]:
# Merging retail with product dataset 

retail = pd.merge(retail,prod, left_on=["prod_cat_code", "prod_subcat_code"],right_on=["prod_cat_code", "prod_sub_cat_code"],how="left")

retail.columns

In [ ]:
# To calculate category preferences by gender,sorted by prod_cat count in descending order

result = retail.groupby(["Gender", "prod_cat"]).size().unstack()  # .unstack() converts gender groups into separate columns.

result.loc["Total"] = result.sum()

result.sort_values(by="Total", axis=1, ascending=False)

In [ ]:
# To calculate sub-category preferences by gender

res = retail.groupby(["Gender","prod_subcat"]).size().unstack()

res.loc["Total"] = res.sum()

res.sort_values(by="Total", axis=1, ascending=False)

#### Conclusion

The gender-wise category analysis shows that **Books is the most frequently purchased product category among both male and female customers**, followed by Electronics.

At the sub-category level, **Women** records the highest number of transaction entries for both genders, with 1,517 for female customers and 1,529 for male customers. Male customers also show a relatively high preference for Mens products (1,502 transaction entries).

Overall, the analysis indicates that male and female customers have similar preferences at the broader category level, with Books being the leading category for both groups.

### <center> Store & Channel Performance </center>

#### 5. Volume vs Value — Which Store Format Wins on Each?

In [ ]:
# Calculating volume vs value excluding negative/return values

purchase = trans[trans["Qty"] > 0]

purchase.groupby("Store_type")[["total_amt", "Qty"]].sum()


In [ ]:
# Calculating volume vs value includind negative/return values

trans.groupby("Store_type")[["total_amt", "Qty"]].sum()

#### Conclusion 

The analysis shows that **e-Shop** leads in both revenue and quantity sold, regardless of whether returns are included or excluded. Among the other store formats, MBR records a slightly higher sales quantity than Flagship stores in both cases.

However, the revenue ranking between MBR and Flagship stores changes depending on the treatment of returns. MBR generates slightly higher revenue when returns are excluded, whereas Flagship stores perform marginally better when returns are included. TeleShop records the lowest revenue and quantity in both comparisons.

Overall, the claim that e-Shop generates the highest revenue is supported, but the claim that Flagship stores sell the highest quantity is not supported by the data.

#### 6. Flagship Revenue Deep-Dive

In [ ]:
flagship = retail[retail["Store_type"] == "Flagship store"]

comparison = pd.concat([flagship.groupby("prod_cat")["total_amt"].sum(),retail.groupby("prod_cat")["total_amt"].sum()],axis=1,
            keys=["Flagship_Revenue", "Total_Revenue"]).loc[["Electronics", "Clothing"]]

comparison["Flagship_Contribution_%"] = (comparison["Flagship_Revenue"] / comparison["Total_Revenue"]) * 100

comparison

#### Conclusion

The analysis compares Flagship store revenue from Electronics and Clothing with the total revenue generated by these categories across all store formats. The Flagship contribution percentage indicates how much each category's overall revenue comes from Flagship stores. This helps assess the importance of Flagship stores in the company's category-wise revenue performance and supports the CFO's annual category P&L analysis.

### <center> Category & Customer Segment Insights </center>

#### 7. The Electronics-Male Segment

In [ ]:
# Total Electronics Revenue from Male customers

male_electronics_revenue = retail[(retail.Gender == "M") & (retail.prod_cat == "Electronics")]["total_amt"].sum().round(3)

print("Total Electronics Revenue from Male customers:",male_electronics_revenue)

# Total Electronics Revenue across all customers

electronics_revenue = retail[retail["prod_cat"] == "Electronics"]["total_amt"].sum().round(3)

print("\nTotal Electronics Revenue:", electronics_revenue)

# To find contribution of male customers in total Electronics Revenue

male_contribution = (male_electronics_revenue / electronics_revenue) * 100

print("\nPercentage of contribution of Male customers in Total Electronics Revenue:",male_contribution.round(2),"%")

#### Conclusion

The analysis shows the contribution of male customers to total Electronics revenue by comparing their spending with the overall revenue generated from Electronics across all customers. This indicates that male customers generate a slightly higher share of Electronics revenue than female customers (46.81%). Therefore, male customers represent the larger revenue-contributing segment in the Electronics category.

#### 8. The Loyal Customers — Who Keeps Coming Back?

In [ ]:
# Excluding return transactions as we want to count genuine customers

purchases = retail[retail["Qty"] > 0]

purchases

In [ ]:
# Counting distinct shopping trips per customer 

customer_trips = purchases.groupby("cust_id")["transaction_id"].nunique()

customer_trips.reset_index()

In [ ]:
# Identifying loyal customers ( who have completed more than 10 distinct shopping trips )

loyal_customers = customer_trips[customer_trips > 10]

loyal_customers.reset_index()

In [ ]:
# Counting total no. of loyal customers

print("Total Loyal Customers:", loyal_customers.count())  # customer_trips.max() = 11

#### Conclusion

The analysis identifies customers who made more than 10 distinct shopping trips during the reporting period, considering only positive-quantity transactions i.e. no return or adjustments. These customers qualify as loyal customers based on their repeat purchase frequency. This helped to  identify most regular shoppers so that they can be treated with loyalty programs, exclusive offers and retention strategies.


### <center> Age Cohort Spending Analysis </center>

#### 9. The 25–35 Cohort — Category Spending Profile

In [ ]:
#Since the transaction data covers 2011–2014, we'll calculate the customer's age at the time of each transaction, rather than using their current age.

retail["Age"] = (retail["tran_date"].dt.year - retail["DOB"].dt.year -
    (
        (retail["tran_date"].dt.month < retail["DOB"].dt.month) |
        ((retail["tran_date"].dt.month == retail["DOB"].dt.month) & (retail["tran_date"].dt.day < retail["DOB"].dt.day))
    ).astype(int)
)

retail.Age.reset_index()


The code calculates the actual age of each customer at the time of their transaction. First, .dt.year extracts the transaction year and birth year,
and subtracting them gives the initial age. However, this may be one year higher if the customer's birthday has not yet occurred in that year. To
handle this, the first condition (transaction month < birth month) checks whether the transaction month is earlier than the birth month. The second
condition checks whether both months are the same (==) but the transaction day is earlier than the birth day. The | (OR) operator combines these
conditions because either situation means the birthday has not occurred yet. Within the second condition, & (AND) ensures that both the month and day
conditions are satisfied. These comparisons return Boolean values (True or False), so .astype(int) converts them into 1 and 0, respectively. Finally,
this value is subtracted from the initial year difference: 1 is subtracted if the birthday has not occurred, otherwise 0 is subtracted, giving the
customer's correct age on the transaction date.

In [ ]:
# Filtering customers aged between 25-35 and Electronics or Books category

age_cat = retail[(retail["Age"].between(25, 35)) & ( retail["prod_cat"].isin(["Electronics","Books"]))]

age_cat[["cust_id", "Age", "prod_cat", "total_amt"]]

In [ ]:
# Calculating category-wise total revenue for this age group ( 25-35) .

age_cat.groupby("prod_cat")["total_amt"].sum().reset_index()

#### Conclusion

The analysis shows that customers aged 25–35 generated higher revenue from Books compared to Electronics. Books contributed approximately ₹10.48 lakh more revenue than Electronics, indicating a stronger spending preference for Books within this age group during the reporting period.

#### 10. A Date-Bound Campaign Effectiveness Window

In [ ]:
# Filtering age-group and campaign period.

campaign = retail[(retail["Age"].between(25, 35)) &(retail["tran_date"] >= "2014-01-01") & (retail["tran_date"] < "2014-03-01")]
# including 1 January 2014 and excluding 1 March 2014 (as asked for first 2 months only)

# Calculating total campaign revenue (excluding returns) 

camp_gross_revenue = campaign[campaign["Qty"] > 0]["total_amt"].sum() 

print("\nCampaign Revenue (excluding returns):",camp_gross_revenue)

# Calculating total campaign revenue (including returns)

camp_net_revenue = campaign["total_amt"].sum()

print("Campaign Revenue(including returns):",camp_net_revenue)

# Calculating difference between gross revene and net revenue.

rev_diff = camp_gross_revenue - camp_net_revenue

print("\nThe difference between net revenue and gross revenue:",rev_diff)

# Calculating percentage impact of returns on campaign

print("\nPercentage impact of returns on campaign:",((rev_diff/camp_gross_revenue)*100).round(2),"%")

#### Conclusion

During January and February 2014 ( the first 2 months of 2014), customers aged 25–35 generated purchase revenue of ₹11,97,848.73. After accounting for returns, the net campaign revenue was ₹10,72,243.38. Returns reduced the revenue by ₹1,25,605.35, approximately 10.49% of the purchase revenue. This indicates that returns had a noticeable impact on the campaign's overall revenue.